In [0]:
# ============================================================
# CONFIGURE ME: set your catalog and schema before running
# ============================================================
catalog = "lrcatalog"
schema  = "lr_genie_demo"

# ============================================================
# Setup - no need to edit below this line
# ============================================================

# Set context
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# Create temp view and DA map of meta keys/values
spark.sql("""
    CREATE OR REPLACE TEMP VIEW user_info AS
    SELECT map_from_arrays(collect_list(key), collect_list(value))
    FROM meta
""")

spark.sql("DECLARE OR REPLACE DA MAP<STRING,STRING>")
spark.sql("SET VAR DA = (SELECT * FROM user_info)")

# Set default schema for the user
spark.sql("USE SCHEMA IDENTIFIER(DA.schema_name)")

# Copy tables from our seed tables
for region, table in [
    ("au", "opportunities"),
    ("au", "orders"),
    ("au", "customers"),
    ("ca", "opportunities"),
    ("ca", "orders"),
    ("ca", "customers"),
]:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {table}_{region} AS
        SELECT * FROM {catalog}.{schema}.{table}_{region}
    """)
    print(f"✅ Created {table}_{region}")

# Show user their catalog + schema
display(spark.sql("SELECT current_catalog() AS CourseCatalog, current_schema() AS YourSchema"))